In [119]:
import mdtraj as md

import nglview as nv

from ipywidgets.embed import embed_data

from typing import List, Optional

import io

import os

from slide_types.SingleColumnSlide import SingleColumnSlide

class NglViewContent:
    def __init__(
        self, 
        pdb_paths: List[str],
        align: bool = True,
        ) -> None:

        self.pdb_paths = pdb_paths
        self.align = align

        self.trajectories = [
            md.load(path, top=path) for path in self.pdb_paths
        ]

        if self.align and len(self.trajectories) > 1:
            self._align_structures()

        self.view = nv.show_mdtraj(self.trajectories[0])
        self.view.clear()

        if len(self.trajectories) == 1:

            self.view.add_cartoon(
                selection="protein",
                color="residueindex"
            )

        else:
            self._build_representations()

    def _align_structures(self) -> None:
        ref_traj = self.trajectories[0]
        ref_ca = ref_traj.topology.select("name CA")

        for ndx, target_traj in enumerate(self.trajectories[1:], start=1):
            target_ca = target_traj.topology.select("name CA")

            target_traj.superpose(
                ref_traj,
                atom_indices=target_ca,
                ref_atom_indices=ref_ca
            )

    def _build_representations(self) -> None:
        self.view.clear()

        for ndx, traj in enumerate(self.trajectories):
            # ADD ME DYNAMICALLY LATER
            #color = self.colors[idx % len(self.colors)]
            component_traj = nv.MDTrajTrajectory(traj)
            component = self.view.add_trajectory(component_traj)
            component.add_cartoon(
                selection="protein"
            )

    def render_html(self) -> str:
        data = embed_data(views=[self.view])

        views_list = data.get('widget_views', data.get('views', []))
        views_html = "".join(views_list)
        return f"""
        <div class="nglview-container" style="width: 100%; min-height: 500px;">
            {views_html}
        </div>
        """

    def render_html2(self) -> str:
        """
        Renders the NGLView widget directly into an in-memory string buffer.
        """
        # Create an in-memory string stream
        buffer = io.StringIO()

        # Write directly to the stream (no disk I/O)
        nv.write_html(buffer, [self.view])
        
        # Extract the HTML string
        full_html = buffer.getvalue()
        buffer.close()

        # Return wrapped inside your slide layout container
        return f"""
        <div class="nglview-container" style="width: 100%; height: 500px; position: relative;">
            {full_html}
        </div>
        """

pdb_dir = "pdb"
pdb_root = "Ab42_seq"
pdb_no = 1000

#######

pdb_name = f"{pdb_root}_{pdb_no}.pdb"
pdb_path = os.path.join(pdb_dir, pdb_name)

ngl_view = NglViewContent(
    pdb_paths=[pdb_path]
)

slide = SingleColumnSlide(
    title="Protein Structure",
    content_block=ngl_view
)

slide.display()

In [120]:
pdb_dir = "pdb"
pdb_root = "Ab42_seq"
pdb_no = 1000

#######

pdb_paths = [
    os.path.join(pdb_dir, i) for i in os.listdir(pdb_dir)
]

print(pdb_paths)
ngl_view = NglViewContent(
    pdb_paths=pdb_paths
)

slide = SingleColumnSlide(
    title="Protein Structure",
    content_block=ngl_view
)

slide.display()

['pdb/Ab42_seq_1000.pdb', 'pdb/Ab42_seq_402.pdb']


In [121]:
!jupyter nbconvert nglviewtest.ipynb --to slides --no-input --output nglviewtest
!brave nglviewtest.slides.html

[NbConvertApp] Converting notebook nglviewtest.ipynb to slides
[NbConvertApp] Writing 270482 bytes to nglviewtest.slides.html
Opening in existing browser session.
